# 8 - Time series

A dated index is structural information. EViews turns it into the workfile
frequency; R and Stata get a real date column; and it comes back intact.

In [ ]:
%load_ext econenv
import econenv, numpy as np, pandas as pd

In [ ]:
idx = pd.period_range('1995Q1', periods=100, freq='Q').to_timestamp()
rng = np.random.default_rng(3)
ts = pd.DataFrame({'gdp': 100 + rng.normal(scale=1.2, size=100).cumsum()}, index=idx)
ts['infl'] = 2 + 0.3 * ts.gdp.diff().fillna(0) + rng.normal(scale=0.5, size=100)
ts.head()

In [ ]:
meta = econenv.transfer.annotate(ts, name='ts')
econenv.transfer.metadata(meta).frequency

## EViews: the index becomes the workfile

In [ ]:
%%eviews -i ts -q
equation eq1.ls infl c d(gdp)

In [ ]:
%eviews_get @pagefreq

In [ ]:
%eviews_get @pagesmpl

`d(gdp)` is meaningful here **because** the page is dated. Push a frame with
no date index and EconEnv warns you that lags and differences will treat the
observations as unordered.

## R: a proper POSIXct column

In [ ]:
%%R -i ts
str(ts)
plot(ts$index, ts$gdp, type = 'l', xlab = '', ylab = 'GDP', main = 'Simulated GDP')

## Stata: %tc, with the conversion reported

In [ ]:
import warnings
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    econenv.push('stata', 'default', ts)
for w in caught:
    print('-', w.message)

In [ ]:
%%stata
list in 1/5
describe